# Main Backtest — Kalman-Filtered Pairs Strategy

End-to-end research notebook. Pulls real prices via `yfinance`, runs the full pipeline,
and produces the artefacts shipped in `results/`.

**Reproducibility:** all randomness in the strategy is deterministic given the data. Re-running with the same date range yields identical numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantpairs.cointegration import engle_granger_test
from quantpairs.backtest import run_backtest
from quantpairs.wfo import walk_forward
from quantpairs.attribution import FamaFrenchAttribution
from quantpairs.data import fetch_prices, fetch_fama_french

PAIR = ("KO", "PEP")
START, END = "2018-01-01", "2024-12-31"

## 1. Cointegration evidence

In [ ]:
prices = fetch_prices(list(PAIR), START, END)
log_y, log_x = np.log(prices[PAIR[0]]), np.log(prices[PAIR[1]])
print(engle_granger_test(log_y, log_x))

## 2. In-sample backtest

In [ ]:
bt = run_backtest(log_y, log_x, cost_bps=2.0)
print(bt.summary())
bt.equity_curve.plot(figsize=(11, 4), title=f"In-sample equity: {PAIR[0]} ~ {PAIR[1]}");

## 3. Walk-forward out-of-sample

In [ ]:
wfo = walk_forward(log_y, log_x, train_size=504, test_size=126, cost_bps=2.0)
wfo.oos_kpis

## 4. Fama-French attribution

In [ ]:
ff = fetch_fama_french(START, END)
attribution = FamaFrenchAttribution.fit(wfo.oos_returns, ff, risk_free=ff['RF'])
attribution.to_frame()

## 5. Honest findings

- In-sample Sharpe is consistently higher than OOS; the gap is a function of cost level and look-back.
- KO/PEP cointegration weakens post-2022 — the model's adaptive β captures this, but signal frequency drops.
- Net-of-cost results dominate any conversation: at 5 bps per trade, the strategy is barely profitable on this pair.
- See `docs/methodology.md` for the full assumptions list.